> Repository edition: run from the repository root using an environment with the dependencies in `requirements.txt`. Machine-specific environment paths have been replaced. Historical outputs are retained except environment checks; they are not evidence of a fresh rerun. See `docs/REPRODUCIBILITY.md`.


# CP_24 — Round 13
## Final black-box optimisation round

This notebook prepares the final query for each of the eight black-box
functions using the initial observations and twelve completed feedback
rounds.

The workflow follows the previous notebooks:

1. Check the Python environment and locate the data.
2. Load and validate the cumulative observations.
3. Review round-12 improvements and prediction errors.
4. Refit the surrogate models and compare candidate queries.
5. Select and validate eight final queries.
6. Reflect on feedback-driven adaptation and the
   exploration–exploitation trade-off.

The final-round selection will consider the remaining evaluation budget:
information gained from these queries cannot guide another submission
within this challenge.

Reinforcement learning provides a conceptual comparison for learning
from feedback and balancing exploration with exploitation. The implemented
method remains Bayesian optimisation; no Q-learning agent, transition
model or self-play system is assumed.

Predictions and acquisition scores remain separate from observed results.

In [ ]:
# CP_24 — Round 13
# Check the Python environment and locate the project data.

import sys
from pathlib import Path

project_path = Path.cwd()

# Use the selected notebook kernel; no machine-specific path is required.
expected_python = Path(sys.executable)

initial_data_path = project_path / "initial_data"
feedback_path = project_path / "week_12"

inputs_path = feedback_path / "inputs.txt"
outputs_path = feedback_path / "outputs.txt"

# Expected round numbers; file contents will be validated next.
completed_rounds = 12
submission_round = 13

correct_kernel = (
    Path(sys.executable).resolve() == expected_python.resolve()
)

print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())
print("Project folder:", project_path)

print("\nSelected interpreter check:", correct_kernel)
print("Project folder found:", project_path.is_dir())
print("Initial data folder found:", initial_data_path.is_dir())
print("Feedback folder found:", feedback_path.is_dir())
print("Cumulative inputs found:", inputs_path.is_file())
print("Cumulative outputs found:", outputs_path.is_file())

print("\nExpected completed rounds:", completed_rounds)
print("Round to prepare:", submission_round)

assert correct_kernel, (
    "Select the notebook kernel corresponding to: "
    f"{expected_python}"
)
assert project_path.is_dir(), (
    f"Project folder not found: {project_path}"
)
assert initial_data_path.is_dir(), (
    f"Initial data folder not found: {initial_data_path}"
)
assert feedback_path.is_dir(), (
    f"Feedback folder not found: {feedback_path}"
)
assert inputs_path.is_file(), (
    f"Inputs file not found: {inputs_path}"
)
assert outputs_path.is_file(), (
    f"Outputs file not found: {outputs_path}"
)

import numpy as np
import pandas as pd
from IPython.display import display

print("\nEnvironment and file checks passed.")
print(
    f"Ready to load and validate the {completed_rounds} feedback rounds."
)

## 2. Load and validate the cumulative observations

The initial observations are combined with twelve completed feedback
rounds for each function. Each round must contain one query and one
observed output for each of the eight functions.

Validation checks the number of rounds, input dimensions, observation
counts, finite numerical values and input bounds. Negative and extremely
small outputs are retained.

The cumulative feedback files are loaded once. Earlier weekly snapshots
are not appended again, avoiding repeated counting of evaluations.

The expected total is 271 evaluated input-output pairs:
175 initial observations and 96 evaluations from twelve rounds.

In [2]:
# CP_24 — Round 13
# Load and validate the initial observations and cumulative feedback.

import ast
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

required_variables = [
    "initial_data_path",
    "inputs_path",
    "outputs_path",
    "completed_rounds",
    "submission_round",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the environment-check cell first. Missing variables: "
        + ", ".join(missing_variables)
    )


def parse_numeric_expression(node):
    """Read numeric literals, lists and array wrappers without eval."""
    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)) and not isinstance(
            node.value, bool
        ):
            return node.value
        raise ValueError("Only numerical constants are permitted.")

    if isinstance(node, (ast.List, ast.Tuple)):
        return [parse_numeric_expression(item) for item in node.elts]

    if isinstance(node, ast.UnaryOp) and isinstance(
        node.op, (ast.UAdd, ast.USub)
    ):
        value = parse_numeric_expression(node.operand)
        if not isinstance(value, (int, float)):
            raise ValueError("A sign must precede a numerical value.")
        return value if isinstance(node.op, ast.UAdd) else -value

    if isinstance(node, ast.Call):
        function = node.func

        allowed_array = (
            isinstance(function, ast.Name)
            and function.id == "array"
        ) or (
            isinstance(function, ast.Attribute)
            and isinstance(function.value, ast.Name)
            and function.value.id in {"np", "numpy"}
            and function.attr == "array"
        )

        if allowed_array and len(node.args) == 1 and not node.keywords:
            return parse_numeric_expression(node.args[0])

    raise ValueError(
        "Unsupported expression in feedback file: "
        + ast.dump(node, include_attributes=False)[:160]
    )


def read_feedback_rounds(path, expected_rounds):
    """Read separate round lists or one outer list containing all rounds."""
    path = Path(path)
    text = path.read_text(encoding="utf-8-sig").strip()

    if not text:
        raise ValueError(f"Feedback file is empty: {path}")

    try:
        tree = ast.parse(text, mode="exec")
    except SyntaxError as error:
        raise ValueError(
            f"Unable to parse feedback file: {path.name}. "
            "Keep its original numerical list format."
        ) from error

    values = []
    for statement in tree.body:
        if not isinstance(statement, ast.Expr):
            raise ValueError(
                f"Unexpected statement in {path.name}; "
                "only numerical round lists are permitted."
            )
        values.append(parse_numeric_expression(statement.value))

    # Accept either twelve separate lists or an outer list of twelve rounds.
    if (
        len(values) == 1
        and isinstance(values[0], list)
        and len(values[0]) == expected_rounds
        and all(
            isinstance(item, list) and len(item) == 8
            for item in values[0]
        )
    ):
        values = values[0]

    if len(values) != expected_rounds:
        raise ValueError(
            f"{path.name}: expected {expected_rounds} rounds, "
            f"found {len(values)}."
        )

    for round_number, round_values in enumerate(values, start=1):
        if not isinstance(round_values, list) or len(round_values) != 8:
            raise ValueError(
                f"{path.name}, round {round_number}: "
                "expected exactly eight function entries."
            )

    return values


function_dimensions = {
    1: 2,
    2: 2,
    3: 3,
    4: 4,
    5: 4,
    6: 5,
    7: 6,
    8: 8,
}

expected_initial_counts = {
    1: 10,
    2: 10,
    3: 15,
    4: 30,
    5: 20,
    6: 20,
    7: 30,
    8: 40,
}

input_rounds = read_feedback_rounds(inputs_path, completed_rounds)
output_rounds = read_feedback_rounds(outputs_path, completed_rounds)

# Rebuild from source files so rerunning this cell cannot append data twice.
initial_data = {}
feedback_data = {}
updated_data = {}
validation_records = []

for function_id in range(1, 9):
    dimensions = function_dimensions[function_id]
    expected_initial = expected_initial_counts[function_id]

    function_folder = (
        Path(initial_data_path) / f"function_{function_id}"
    )
    initial_inputs_file = function_folder / "initial_inputs.npy"
    initial_outputs_file = function_folder / "initial_outputs.npy"

    for file_path in [initial_inputs_file, initial_outputs_file]:
        if not file_path.is_file():
            raise FileNotFoundError(f"Required file not found: {file_path}")

    X_initial = np.asarray(
        np.load(initial_inputs_file, allow_pickle=False),
        dtype=float,
    )
    y_initial_raw = np.asarray(
        np.load(initial_outputs_file, allow_pickle=False),
        dtype=float,
    )

    if X_initial.shape != (expected_initial, dimensions):
        raise ValueError(
            f"Function {function_id}: initial input shape "
            f"{X_initial.shape}; expected "
            f"({expected_initial}, {dimensions})."
        )

    if y_initial_raw.shape not in {
        (expected_initial,),
        (expected_initial, 1),
    }:
        raise ValueError(
            f"Function {function_id}: unexpected initial output shape "
            f"{y_initial_raw.shape}."
        )

    y_initial = y_initial_raw.reshape(-1)

    function_inputs = []
    function_outputs = []

    for round_index in range(completed_rounds):
        query = np.asarray(
            input_rounds[round_index][function_id - 1],
            dtype=float,
        )
        output = np.asarray(
            output_rounds[round_index][function_id - 1],
            dtype=float,
        )

        if query.shape != (dimensions,):
            raise ValueError(
                f"Function {function_id}, round {round_index + 1}: "
                f"input shape {query.shape}; expected ({dimensions},)."
            )

        if output.size != 1:
            raise ValueError(
                f"Function {function_id}, round {round_index + 1}: "
                "expected one scalar output."
            )

        function_inputs.append(query)
        function_outputs.append(float(output.reshape(-1)[0]))

    X_feedback = np.vstack(function_inputs)
    y_feedback = np.asarray(function_outputs, dtype=float)

    X_all = np.vstack([X_initial, X_feedback])
    y_all = np.concatenate([y_initial, y_feedback])

    if not np.all(np.isfinite(X_all)):
        raise ValueError(
            f"Function {function_id}: non-finite input coordinates."
        )

    if not np.all(np.isfinite(y_all)):
        raise ValueError(
            f"Function {function_id}: non-finite observed outputs."
        )

    # Validate historical inputs against the unit-cube domain.
    if np.any(X_all < 0.0) or np.any(X_all > 1.0):
        raise ValueError(
            f"Function {function_id}: historical inputs outside [0, 1]."
        )

    expected_total = expected_initial + completed_rounds

    if X_all.shape != (expected_total, dimensions):
        raise ValueError(
            f"Function {function_id}: incorrect cumulative input shape."
        )

    if y_all.shape != (expected_total,):
        raise ValueError(
            f"Function {function_id}: incorrect cumulative output shape."
        )

    initial_data[function_id] = {
        "X": X_initial.copy(),
        "y": y_initial.copy(),
    }
    feedback_data[function_id] = {
        "X": X_feedback.copy(),
        "y": y_feedback.copy(),
    }
    updated_data[function_id] = {
        "X": X_all.copy(),
        "y": y_all.copy(),
    }

    validation_records.append({
        "Function": function_id,
        "Initial observations": expected_initial,
        "Feedback observations": completed_rounds,
        "Total observations": len(y_all),
        "Dimensions": dimensions,
    })

    print(
        f"Function {function_id}: "
        f"{expected_initial} → {len(y_all)} observations "
        f"| {dimensions} variables"
    )

data_validation_df = (
    pd.DataFrame(validation_records)
    .set_index("Function")
)

total_observations = sum(
    len(dataset["y"]) for dataset in updated_data.values()
)
expected_observations = (
    sum(expected_initial_counts.values()) + 8 * completed_rounds
)

assert total_observations == expected_observations
assert submission_round == completed_rounds + 1

print(f"\nValidated feedback rounds: {completed_rounds}")
print(f"Round to prepare: {submission_round}")
print(f"Total evaluated observations: {total_observations}")
print("All eight datasets loaded and validated successfully.")

Function 1: 10 → 22 observations | 2 variables
Function 2: 10 → 22 observations | 2 variables
Function 3: 15 → 27 observations | 3 variables
Function 4: 30 → 42 observations | 4 variables
Function 5: 20 → 32 observations | 4 variables
Function 6: 20 → 32 observations | 5 variables
Function 7: 30 → 42 observations | 6 variables
Function 8: 40 → 52 observations | 8 variables

Validated feedback rounds: 12
Round to prepare: 13
Total evaluated observations: 271
All eight datasets loaded and validated successfully.


## 3. Review observed improvements through round 12

Each round's observed output is compared with the best value available
before that evaluation. All eight functions are maximisation problems,
including those with negative outputs.

The analysis reports the latest results and the improvement history.
An improvement is counted only when an observed output strictly exceeds
the previous best. Negative and extremely small values are preserved.

Unsuccessful queries remain in the training data. They can inform future
predictions even when they do not improve the best observed result.

This feedback provides a conceptual link to reinforcement learning:
new outcomes revise expectations and inform subsequent decisions.
However, the calculations below track observed optimisation progress;
they do not implement Q-learning or estimate cumulative rewards.

In [3]:
# CP_24 — Round 13
# Review observed improvements through the twelve completed rounds.

import numpy as np
import pandas as pd
from IPython.display import display

required_variables = [
    "initial_data",
    "feedback_data",
    "updated_data",
    "completed_rounds",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the data-loading cell first. Missing variables: "
        + ", ".join(missing_variables)
    )

improvement_records = []
latest_round_records = []
overall_records = []

for function_id in range(1, 9):
    y_initial = np.asarray(
        initial_data[function_id]["y"], dtype=float
    ).reshape(-1)

    y_feedback = np.asarray(
        feedback_data[function_id]["y"], dtype=float
    ).reshape(-1)

    y_all = np.asarray(
        updated_data[function_id]["y"], dtype=float
    ).reshape(-1)

    if len(y_feedback) != completed_rounds:
        raise ValueError(
            f"Function {function_id}: expected "
            f"{completed_rounds} feedback outputs, "
            f"found {len(y_feedback)}."
        )

    initial_best = float(np.max(y_initial))
    best_so_far = initial_best
    improving_rounds = 0

    for round_number, observed_output in enumerate(
        y_feedback, start=1
    ):
        observed_output = float(observed_output)
        best_before = best_so_far

        # Strict comparison preserves tiny numerical improvements.
        improved = observed_output > best_before
        improvement = (
            observed_output - best_before if improved else 0.0
        )

        best_so_far = max(best_before, observed_output)
        improving_rounds += int(improved)

        improvement_records.append({
            "Function": function_id,
            "Round": round_number,
            "Best before round": best_before,
            "Observed output": observed_output,
            "Improved": improved,
            "Improvement": improvement,
            "Best so far": best_so_far,
        })

        if round_number == completed_rounds:
            latest_round_records.append({
                "Function": function_id,
                f"Best before round {completed_rounds}": best_before,
                f"Round {completed_rounds} output": observed_output,
                f"Round {completed_rounds} improved": improved,
                "Improvement": improvement,
                "Current best": best_so_far,
            })

    # Confirm that sequential tracking agrees with the full dataset.
    if best_so_far != float(np.max(y_all)):
        raise ValueError(
            f"Function {function_id}: best-so-far tracking "
            "does not match the cumulative dataset."
        )

    overall_records.append({
        "Function": function_id,
        "Observations": len(y_all),
        "Initial best": initial_best,
        "Current best": best_so_far,
        "Improvement over initial best": best_so_far - initial_best,
        "Number of improving rounds": improving_rounds,
    })

improvement_history_df = pd.DataFrame(improvement_records)

round_comparison_df = (
    pd.DataFrame(latest_round_records)
    .set_index("Function")
    .sort_index()
)

overall_progress_df = (
    pd.DataFrame(overall_records)
    .set_index("Function")
    .sort_index()
)

with pd.option_context(
    "display.max_columns", None,
    "display.float_format", "{:.6e}".format,
):
    display(round_comparison_df)

for round_number in range(1, completed_rounds + 1):
    improved_functions = improvement_history_df.loc[
        (improvement_history_df["Round"] == round_number)
        & improvement_history_df["Improved"],
        "Function",
    ].tolist()

    print(
        f"Functions improved in round {round_number}: "
        f"{improved_functions} | Count: {len(improved_functions)}"
    )

print("\nOverall progress from the initial observations")

with pd.option_context(
    "display.max_columns", None,
    "display.float_format", "{:.6e}".format,
):
    display(overall_progress_df)

print(
    "\nImprovements are calculated from observed outputs, "
    "not model predictions."
)
print(
    "Output scales differ across functions; "
    "raw improvement magnitudes should not be compared across functions."
)
print(
    "A round without improvement does not establish convergence "
    "or prove that its evaluations provided no useful information."
)

,Best before round 12,Round 12 output,Round 12 improved,Improvement,Current best
Function,,,,,
1,7.710875e-16,2.868521e-134,False,0.000000e+00,7.710875e-16
2,6.582395e-01,5.569841e-01,False,0.000000e+00,6.582395e-01
3,-3.096391e-02,-5.428389e-02,False,0.000000e+00,-3.096391e-02
4,6.152729e-01,5.421303e-01,False,0.000000e+00,6.152729e-01
5,8.515379e+03,8.363044e+03,False,0.000000e+00,8.515379e+03
6,-2.424310e-01,-4.032869e-01,False,0.000000e+00,-2.424310e-01
7,2.436323e+00,5.281285e-01,False,0.000000e+00,2.436323e+00
8,9.958227e+00,9.939518e+00,False,0.000000e+00,9.958227e+00


Functions improved in round 1: [2, 4, 5, 7, 8] | Count: 5
Functions improved in round 2: [6, 8] | Count: 2
Functions improved in round 3: [5, 8] | Count: 2
Functions improved in round 4: [6, 8] | Count: 2
Functions improved in round 5: [3, 4, 5, 8] | Count: 4
Functions improved in round 6: [5, 6, 7] | Count: 3
Functions improved in round 7: [5] | Count: 1
Functions improved in round 8: [4, 5, 7] | Count: 3
Functions improved in round 9: [8] | Count: 1
Functions improved in round 10: [5, 7] | Count: 2
Functions improved in round 11: [] | Count: 0
Functions improved in round 12: [] | Count: 0

Overall progress from the initial observations


,Observations,Initial best,Current best,Improvement over initial best,Number of improving rounds
Function,,,,,
1,22,7.710875e-16,7.710875e-16,0.000000e+00,0
2,22,6.112052e-01,6.582395e-01,4.703429e-02,1
3,27,-3.483531e-02,-3.096391e-02,3.871400e-03,1
4,42,-4.025542e+00,6.152729e-01,4.640815e+00,3
5,32,1.088860e+03,8.515379e+03,7.426519e+03,7
6,32,-7.142649e-01,-2.424310e-01,4.718340e-01,3
7,42,1.364968e+00,2.436323e+00,1.071355e+00,4
8,52,9.598482e+00,9.958227e+00,3.597446e-01,6



Improvements are calculated from observed outputs, not model predictions.
Output scales differ across functions; raw improvement magnitudes should not be compared across functions.
A round without improvement does not establish convergence or prove that its evaluations provided no useful information.


## 4. Compare round-12 predictions with observed outputs

Neither round 11 nor round 12 improved the best observed values.
However, seven functions remain above their initial best results.
This recent lack of improvement does not establish convergence.

The predictions recorded in CP_23 are compared with the actual round-12
outputs. Historical query coordinates are checked against the feedback
inputs before calculating prediction errors.

Function 7 requires particular attention: its global proposal returned
approximately 0.528, compared with a GP prediction of approximately
2.217. This discrepancy challenges confidence in that prediction and
motivates reviewing the final-round exploration policy.

Function 5's distance-constrained queries did not improve the incumbent
in rounds 11 or 12. The constraint will therefore be reconsidered,
without assuming that the unevaluated alternatives would have succeeded.

Absolute error divided by predictive standard deviation is an
observation-level diagnostic, not a complete uncertainty calibration
assessment. Historical predictions are rounded, so errors are approximate.

These comparisons use predictions made before the outcomes were known.
Refitting a model on the new observations would not provide an equivalent
assessment of its original predictive performance.

In [4]:
# CP_24 — Round 13
# Compare historical CP_23 predictions with round-12 feedback.

import numpy as np
import pandas as pd
from IPython.display import display

required_variables = ["feedback_data", "completed_rounds"]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the data-loading cell first. Missing variables: "
        + ", ".join(missing_variables)
    )

historical_round = 12

if completed_rounds != historical_round:
    raise ValueError(
        "This cell expects twelve completed feedback rounds."
    )

# Rounded predictions recorded in CP_23 before round-12 feedback.
historical_predictions = {
    1: {
        "query": "0.000794-0.306216",
        "mean": -5.191403e-05,
        "std": 6.765328e-04,
        "auxiliary": np.nan,
    },
    2: {
        "query": "0.695216-0.000340",
        "mean": 5.687673e-01,
        "std": 7.315649e-02,
        "auxiliary": np.nan,
    },
    3: {
        "query": "0.009599-0.996723-0.489470",
        "mean": -3.609319e-02,
        "std": 4.468820e-02,
        "auxiliary": np.nan,
    },
    4: {
        "query": "0.401292-0.414900-0.382216-0.419184",
        "mean": 4.382956e-01,
        "std": 1.750837e-01,
        "auxiliary": np.nan,
    },
    5: {
        "query": "0.986568-0.998687-0.999548-0.999338",
        "mean": 8206.489417,
        "std": 255.473602,
        "auxiliary": 8146.865978,
    },
    6: {
        "query": "0.474194-0.394624-0.550990-0.711773-0.003309",
        "mean": -3.566373e-01,
        "std": 9.513187e-02,
        "auxiliary": np.nan,
    },
    7: {
        "query": (
            "0.023970-0.738916-0.861002-"
            "0.151043-0.315073-0.719810"
        ),
        "mean": 2.216998,
        "std": 1.695343e-01,
        "auxiliary": 1.286454,
    },
    8: {
        "query": (
            "0.147547-0.091396-0.058935-0.207967-"
            "0.996495-0.564027-0.249450-0.839157"
        ),
        "mean": 9.961708,
        "std": 1.754157e-02,
        "auxiliary": np.nan,
    },
}

prediction_error_records = []

for function_id in range(1, 9):
    historical = historical_predictions[function_id]

    historical_query = np.array(
        [float(value) for value in historical["query"].split("-")],
        dtype=float,
    )

    feedback_inputs = np.asarray(
        feedback_data[function_id]["X"], dtype=float
    )
    feedback_outputs = np.asarray(
        feedback_data[function_id]["y"], dtype=float
    ).reshape(-1)

    if (
        len(feedback_inputs) != completed_rounds
        or len(feedback_outputs) != completed_rounds
    ):
        raise ValueError(
            f"Function {function_id}: unexpected feedback length."
        )

    observed_query = feedback_inputs[historical_round - 1]
    observed_output = float(
        feedback_outputs[historical_round - 1]
    )

    if historical_query.shape != observed_query.shape:
        raise ValueError(
            f"Function {function_id}: historical query dimensions "
            "do not match the feedback."
        )

    if not np.allclose(
        historical_query,
        observed_query,
        rtol=0.0,
        atol=5e-7,
    ):
        raise ValueError(
            f"Function {function_id}: the CP_23 query does not match "
            "the round-12 feedback input. Check the files before "
            "interpreting prediction errors."
        )

    predicted_mean = float(historical["mean"])
    predicted_std = float(historical["std"])
    auxiliary_prediction = float(historical["auxiliary"])

    if not np.all(np.isfinite([
        predicted_mean,
        predicted_std,
        observed_output,
    ])):
        raise ValueError(
            f"Function {function_id}: non-finite prediction or output."
        )

    if predicted_std <= 0:
        raise ValueError(
            f"Function {function_id}: predicted standard deviation "
            "must be positive."
        )

    residual = observed_output - predicted_mean
    absolute_error = abs(residual)
    error_over_std = absolute_error / predicted_std

    auxiliary_absolute_error = (
        abs(observed_output - auxiliary_prediction)
        if np.isfinite(auxiliary_prediction)
        else np.nan
    )

    prediction_error_records.append({
        "Function": function_id,
        "Predicted mean": predicted_mean,
        "Predicted std": predicted_std,
        "Observed output": observed_output,
        "Observed minus predicted": residual,
        "Absolute error / predicted std": error_over_std,
        "GP absolute error": absolute_error,
        "Auxiliary prediction": auxiliary_prediction,
        "Auxiliary absolute error": auxiliary_absolute_error,
    })

historical_round12 = (
    pd.DataFrame.from_dict(historical_predictions, orient="index")
    .rename_axis("Function")
    .sort_index()
)

prediction_errors_df = (
    pd.DataFrame(prediction_error_records)
    .set_index("Function")
    .sort_index()
)

scientific_columns = [
    column for column in prediction_errors_df.columns
    if column != "Absolute error / predicted std"
]

formatters = {
    column: "{:.6e}" for column in scientific_columns
}
formatters["Absolute error / predicted std"] = "{:.2f}"

display(
    prediction_errors_df.style.format(
        formatters,
        na_rep="—",
    )
)

print(
    "\nAll eight historical queries match the round 12 feedback inputs."
)
print("Errors use rounded historical predictions from CP_23.")
print(
    "Error / predicted std is a diagnostic for these individual "
    "observations, not a complete uncertainty calibration assessment."
)

,Predicted mean,Predicted std,Observed output,Observed minus predicted,Absolute error / predicted std,GP absolute error,Auxiliary prediction,Auxiliary absolute error
Function,,,,,,,,
1,-5.191403e-05,6.765328e-04,2.868521e-134,5.191403e-05,0.08,5.191403e-05,—,—
2,5.687673e-01,7.315649e-02,5.569841e-01,-1.178324e-02,0.16,1.178324e-02,—,—
3,-3.609319e-02,4.468820e-02,-5.428389e-02,-1.819070e-02,0.41,1.819070e-02,—,—
4,4.382956e-01,1.750837e-01,5.421303e-01,1.038347e-01,0.59,1.038347e-01,—,—
5,8.206489e+03,2.554736e+02,8.363044e+03,1.565547e+02,0.61,1.565547e+02,8.146866e+03,2.161781e+02
6,-3.566373e-01,9.513187e-02,-4.032869e-01,-4.664962e-02,0.49,4.664962e-02,—,—
7,2.216998e+00,1.695343e-01,5.281285e-01,-1.688869e+00,9.96,1.688869e+00,1.286454e+00,7.583255e-01
8,9.961708e+00,1.754157e-02,9.939518e+00,-2.218955e-02,1.26,2.218955e-02,—,—



All eight historical queries match the round 12 feedback inputs.
Errors use rounded historical predictions from CP_23.
Error / predicted std is a diagnostic for these individual observations, not a complete uncertainty calibration assessment.


## 5. Refit Gaussian Process models through round 12

A separate Gaussian Process is fitted to each function using its initial
observations and all twelve completed feedback rounds.

The Matérn smoothness settings and parameter bounds are retained from
CP_23. Continuous kernel parameters are re-estimated by maximising the
log-marginal likelihood, with three optimiser restarts. This step does
not perform a new cross-validation search.

All original input coordinates are retained. The PCA analysis in CP_23
described sampled input variation and did not justify automatically
removing coordinates.

Function 7's round-12 prediction error was approximately 9.96 times its
predicted standard deviation. This indicates a substantial mismatch at
that evaluated point, rather than establishing a general calibration
result. The new observation is included when refitting the model.

Fitting warnings are recorded. Parameters reaching their bounds may
indicate sensitivity to modelling assumptions; they do not automatically
justify changing those bounds.

The fitted white-noise component can absorb noise or model mismatch.
It does not prove that the underlying functions are stochastic.
Predictive uncertainty therefore requires caution when comparing
final-round candidates.

In [5]:
# CP_24 — Round 13
# Refit Gaussian Processes using observations through round 12.

import warnings
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel,
)

required_variables = [
    "updated_data",
    "completed_rounds",
    "submission_round",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the preceding notebook cells first. Missing variables: "
        + ", ".join(missing_variables)
    )

if completed_rounds != 12 or submission_round != 13:
    raise ValueError(
        "This cell expects twelve completed rounds "
        "and preparation for round 13."
    )

# Retain the configurations used in CP_23.
gp_configurations = {
    1: {
        "name": "Matern_1.5_original",
        "nu": 1.5,
        "upper_bound": 2.0,
    },
    2: {
        "name": "Matern_1.5_wider",
        "nu": 1.5,
        "upper_bound": 10.0,
    },
    3: {
        "name": "Matern_1.5_original",
        "nu": 1.5,
        "upper_bound": 2.0,
    },
    4: {
        "name": "Matern_1.5_wider",
        "nu": 1.5,
        "upper_bound": 10.0,
    },
    5: {
        "name": "Matern_1.5_original",
        "nu": 1.5,
        "upper_bound": 2.0,
    },
    6: {
        "name": "Matern_2.5_original",
        "nu": 2.5,
        "upper_bound": 2.0,
    },
    7: {
        "name": "Matern_2.5_original",
        "nu": 2.5,
        "upper_bound": 2.0,
    },
    8: {
        "name": "Matern_2.5_wider",
        "nu": 2.5,
        "upper_bound": 10.0,
    },
}

gp_models = {}
gp_fit_records = []
gp_warning_records = []

for function_id in range(1, 9):
    X = np.asarray(
        updated_data[function_id]["X"], dtype=float
    )
    y = np.asarray(
        updated_data[function_id]["y"], dtype=float
    ).reshape(-1)

    if X.ndim != 2 or X.shape[0] != len(y):
        raise ValueError(
            f"Function {function_id}: incompatible input/output shapes."
        )

    if not np.all(np.isfinite(X)) or not np.all(np.isfinite(y)):
        raise ValueError(
            f"Function {function_id}: non-finite training values."
        )

    dimensions = X.shape[1]
    configuration = gp_configurations[function_id]

    kernel = (
        ConstantKernel(
            constant_value=1.0,
            constant_value_bounds=(1e-3, 1e3),
        )
        * Matern(
            length_scale=np.full(dimensions, 0.2),
            length_scale_bounds=(
                0.02,
                configuration["upper_bound"],
            ),
            nu=configuration["nu"],
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1),
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        alpha=1e-10,
        normalize_y=True,
        n_restarts_optimizer=3,
        random_state=42,
    )

    start_time = perf_counter()

    with warnings.catch_warnings(record=True) as captured_warnings:
        warnings.simplefilter("always")
        model.fit(X, y)

    fit_seconds = perf_counter() - start_time

    gp_models[function_id] = model

    gp_fit_records.append({
        "Function": function_id,
        "Observations": len(y),
        "GP configuration": configuration["name"],
        "Fitted kernel": str(model.kernel_),
        "Log-marginal likelihood": float(
            model.log_marginal_likelihood_value_
        ),
        "GP fit seconds": fit_seconds,
    })

    for warning in captured_warnings:
        gp_warning_records.append({
            "Function": function_id,
            "Category": warning.category.__name__,
            "Message": str(warning.message),
        })

gp_fit_summary_df = (
    pd.DataFrame(gp_fit_records)
    .set_index("Function")
    .sort_index()
)

gp_warnings_df = pd.DataFrame(
    gp_warning_records,
    columns=["Function", "Category", "Message"],
)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_columns", None,
):
    display(
        gp_fit_summary_df.style.format({
            "Log-marginal likelihood": "{:.6f}",
            "GP fit seconds": "{:.3f}",
        })
    )

print(
    f"\nAll eight Gaussian Processes were fitted using "
    f"observations through round {completed_rounds}."
)
print(
    "Kernel configurations were retained from CP_23; "
    "continuous kernel parameters were re-estimated."
)
print("Models use all original input coordinates, not PCA scores.")
print(f"\nRecorded fitting warnings: {len(gp_warnings_df)}")

if not gp_warnings_df.empty:
    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None,
    ):
        display(gp_warnings_df)

,Observations,GP configuration,Fitted kernel,Log-marginal likelihood,GP fit seconds
Function,,,,,
1,22,Matern_1.5_original,"1.09**2 * Matern(length_scale=[2, 0.0418], nu=1.5) + WhiteKernel(noise_level=2.13e-08)",-29.289565,0.431
2,22,Matern_1.5_wider,"1.14**2 * Matern(length_scale=[0.116, 10], nu=1.5) + WhiteKernel(noise_level=0.0667)",-17.699600,0.116
3,27,Matern_1.5_original,"1.17**2 * Matern(length_scale=[2, 1.49, 0.0747], nu=1.5) + WhiteKernel(noise_level=2.54e-09)",-31.516682,0.192
4,42,Matern_1.5_wider,"4.42**2 * Matern(length_scale=[3.68, 3.34, 3.51, 3.6], nu=1.5) + WhiteKernel(noise_level=0.000188)",11.264367,0.440
5,32,Matern_1.5_original,"2.5**2 * Matern(length_scale=[2, 1.93, 1.93, 1.96], nu=1.5) + WhiteKernel(noise_level=1e-10)",2.696426,0.273
6,32,Matern_2.5_original,"1.68**2 * Matern(length_scale=[1.03, 1.33, 1.42, 1.25, 2], nu=2.5) + WhiteKernel(noise_level=0.0199)",-18.039983,0.274
7,42,Matern_2.5_original,"0.626**2 * Matern(length_scale=[0.856, 0.394, 2, 0.219, 0.212, 0.238], nu=2.5) + WhiteKernel(noise_level=6.21e-05)",-26.433327,0.194
8,52,Matern_2.5_wider,"3.27**2 * Matern(length_scale=[3.54, 5.3, 2.85, 5.14, 10, 5.24, 3.61, 10], nu=2.5) + WhiteKernel(noise_level=1.63e-05)",24.282439,0.304



All eight Gaussian Processes were fitted using observations through round 12.
Kernel configurations were retained from CP_23; continuous kernel parameters were re-estimated.
Models use all original input coordinates, not PCA scores.

Recorded fitting warnings: 11


,Function,Category,Message
0,1,ConvergenceWarning,The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
1,2,ConvergenceWarning,The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
2,3,ConvergenceWarning,The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
3,5,ConvergenceWarning,lbfgs failed to converge after 24 iteration(s) (status=2):\nABNORMAL: \n\nYou might also want to scale the data as shown in:\n https://scikit-learn.org/stable/modules/preprocessing.html
4,5,ConvergenceWarning,lbfgs failed to converge after 27 iteration(s) (status=2):\nABNORMAL: \n\nYou might also want to scale the data as shown in:\n https://scikit-learn.org/stable/modules/preprocessing.html
5,5,ConvergenceWarning,The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
6,5,ConvergenceWarning,The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
7,6,ConvergenceWarning,The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
8,7,ConvergenceWarning,The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
9,8,ConvergenceWarning,The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.


### 5.1. Check optimisation stability for Function 5

Function 5 produced two L-BFGS convergence warnings. These differ from
warnings about kernel parameters reaching their bounds: some optimisation
runs terminated abnormally.

I compare the existing fit with two additional fits using the same data,
kernel family, bounds and target normalisation, but different random
seeds and ten optimiser restarts.

The comparison records log-marginal likelihood, fitted kernels and
warnings. Predictions are also compared on a shared, reproducible probe
set containing global points and points near the best observation.

Prediction differences measure sensitivity to the fitting procedure.
The probe points have no observed outputs, so this is not a predictive
accuracy test. A higher training likelihood does not by itself establish
better optimisation performance.

The existing model is retained until these diagnostics have been reviewed.

In [6]:
# CP_24 — Round 13
# Check Function 5 fitting stability without replacing the current model.

import warnings
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.gaussian_process import GaussianProcessRegressor

required_variables = [
    "updated_data",
    "gp_models",
    "gp_warnings_df",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the GP-fitting cell first. Missing variables: "
        + ", ".join(missing_variables)
    )

function_id = 5

X_f5 = np.asarray(updated_data[function_id]["X"], dtype=float)
y_f5 = np.asarray(
    updated_data[function_id]["y"], dtype=float
).reshape(-1)

reference_model = gp_models[function_id]
best_input = X_f5[np.argmax(y_f5)]

# Shared probe set for comparing model predictions only.
probe_rng = np.random.default_rng(2405)

global_probes = probe_rng.uniform(
    0.0, 1.0, size=(1000, X_f5.shape[1])
)

local_lower = np.maximum(0.0, best_input - 0.03)
local_upper = np.minimum(1.0, best_input + 0.03)

local_probes = probe_rng.uniform(
    local_lower,
    local_upper,
    size=(1000, X_f5.shape[1]),
)

f5_probe_points = np.vstack([global_probes, local_probes])
reference_mean = reference_model.predict(f5_probe_points)

# Retain all alternatives for review.
f5_stability_models = {"Current fit": reference_model}
stability_records = []
stability_warning_records = []

reference_warnings = gp_warnings_df.loc[
    gp_warnings_df["Function"] == function_id
]

stability_records.append({
    "Fit": "Current fit",
    "Seed": reference_model.random_state,
    "Optimiser restarts": reference_model.n_restarts_optimizer,
    "Log-marginal likelihood": float(
        reference_model.log_marginal_likelihood_value_
    ),
    "LML difference from current": 0.0,
    "Global mean difference RMSE": 0.0,
    "Local mean difference RMSE": 0.0,
    "Maximum absolute mean difference": 0.0,
    "Warnings": len(reference_warnings),
    "L-BFGS convergence warnings": sum(
        "failed to converge" in str(message).lower()
        for message in reference_warnings["Message"]
    ),
    "Fitted kernel": str(reference_model.kernel_),
    "Fit seconds": np.nan,
})

for seed in [42, 137]:
    label = f"10 restarts / seed {seed}"

    # Clone the initial kernel specification, not the fitted kernel.
    alternative_model = GaussianProcessRegressor(
        kernel=clone(reference_model.kernel),
        alpha=reference_model.alpha,
        normalize_y=reference_model.normalize_y,
        n_restarts_optimizer=10,
        random_state=seed,
    )

    start_time = perf_counter()

    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always")
        alternative_model.fit(X_f5, y_f5)

    fit_seconds = perf_counter() - start_time

    alternative_mean, alternative_std = alternative_model.predict(
        f5_probe_points,
        return_std=True,
    )

    if (
        not np.all(np.isfinite(alternative_mean))
        or not np.all(np.isfinite(alternative_std))
        or np.any(alternative_std < 0)
        or not np.isfinite(
            alternative_model.log_marginal_likelihood_value_
        )
    ):
        raise ValueError(
            f"Invalid predictive values or likelihood for {label}."
        )

    differences = alternative_mean - reference_mean

    f5_stability_models[label] = alternative_model

    convergence_warning_count = sum(
        "failed to converge" in str(item.message).lower()
        for item in captured
    )

    stability_records.append({
        "Fit": label,
        "Seed": seed,
        "Optimiser restarts": 10,
        "Log-marginal likelihood": float(
            alternative_model.log_marginal_likelihood_value_
        ),
        "LML difference from current": float(
            alternative_model.log_marginal_likelihood_value_
            - reference_model.log_marginal_likelihood_value_
        ),
        "Global mean difference RMSE": float(
            np.sqrt(np.mean(differences[:1000] ** 2))
        ),
        "Local mean difference RMSE": float(
            np.sqrt(np.mean(differences[1000:] ** 2))
        ),
        "Maximum absolute mean difference": float(
            np.max(np.abs(differences))
        ),
        "Warnings": len(captured),
        "L-BFGS convergence warnings": convergence_warning_count,
        "Fitted kernel": str(alternative_model.kernel_),
        "Fit seconds": fit_seconds,
    })

    for item in captured:
        stability_warning_records.append({
            "Fit": label,
            "Category": item.category.__name__,
            "Message": str(item.message),
        })

f5_stability_df = (
    pd.DataFrame(stability_records)
    .set_index("Fit")
)

f5_stability_warnings_df = pd.DataFrame(
    stability_warning_records,
    columns=["Fit", "Category", "Message"],
)

numeric_formats = {
    "Log-marginal likelihood": "{:.8f}",
    "LML difference from current": "{:.8f}",
    "Global mean difference RMSE": "{:.6f}",
    "Local mean difference RMSE": "{:.6f}",
    "Maximum absolute mean difference": "{:.6f}",
    "Fit seconds": "{:.3f}",
}

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_columns", None,
):
    display(
        f5_stability_df.style.format(
            numeric_formats,
            na_rep="—",
        )
    )

if not f5_stability_warnings_df.empty:
    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None,
    ):
        display(f5_stability_warnings_df)

print("\nFunction 5 stability checks completed.")
print("Prediction differences are reported in original output units.")
print(
    "Probe points have no observed outputs; these differences "
    "are not predictive accuracy measurements."
)
print(
    "Warnings may originate from individual optimiser starts, "
    "not necessarily the retained best solution."
)
print("The current model in gp_models[5] has not been replaced.")

,Seed,Optimiser restarts,Log-marginal likelihood,LML difference from current,Global mean difference RMSE,Local mean difference RMSE,Maximum absolute mean difference,Warnings,L-BFGS convergence warnings,Fitted kernel,Fit seconds
Fit,,,,,,,,,,,
Current fit,42,3,2.69642620,0.00000000,0.000000,0.000000,0.000000,4,2,"2.5**2 * Matern(length_scale=[2, 1.93, 1.93, 1.96], nu=1.5) + WhiteKernel(noise_level=1e-10)",—
10 restarts / seed 42,42,10,2.69642627,0.00000007,0.002004,0.000243,0.007363,6,4,"2.5**2 * Matern(length_scale=[2, 1.93, 1.93, 1.96], nu=1.5) + WhiteKernel(noise_level=1e-10)",1.256
10 restarts / seed 137,137,10,2.69642638,0.00000019,0.005617,0.000296,0.018816,6,4,"2.5**2 * Matern(length_scale=[2, 1.93, 1.93, 1.96], nu=1.5) + WhiteKernel(noise_level=1e-10)",1.528


,Fit,Category,Message
0,10 restarts / seed 42,ConvergenceWarning,lbfgs failed to converge after 24 iteration(s) (status=2):\nABNORMAL: \n\nYou might also want to scale the data as shown in:\n https://scikit-learn.org/stable/modules/preprocessing.html
1,10 restarts / seed 42,ConvergenceWarning,lbfgs failed to converge after 27 iteration(s) (status=2):\nABNORMAL: \n\nYou might also want to scale the data as shown in:\n https://scikit-learn.org/stable/modules/preprocessing.html
2,10 restarts / seed 42,ConvergenceWarning,lbfgs failed to converge after 28 iteration(s) (status=2):\nABNORMAL: \n\nYou might also want to scale the data as shown in:\n https://scikit-learn.org/stable/modules/preprocessing.html
3,10 restarts / seed 42,ConvergenceWarning,lbfgs failed to converge after 25 iteration(s) (status=2):\nABNORMAL: \n\nYou might also want to scale the data as shown in:\n https://scikit-learn.org/stable/modules/preprocessing.html
4,10 restarts / seed 42,ConvergenceWarning,The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
5,10 restarts / seed 42,ConvergenceWarning,The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
6,10 restarts / seed 137,ConvergenceWarning,lbfgs failed to converge after 25 iteration(s) (status=2):\nABNORMAL: \n\nYou might also want to scale the data as shown in:\n https://scikit-learn.org/stable/modules/preprocessing.html
7,10 restarts / seed 137,ConvergenceWarning,lbfgs failed to converge after 20 iteration(s) (status=2):\nABNORMAL: \n\nYou might also want to scale the data as shown in:\n https://scikit-learn.org/stable/modules/preprocessing.html
8,10 restarts / seed 137,ConvergenceWarning,lbfgs failed to converge after 26 iteration(s) (status=2):\nABNORMAL: \n\nYou might also want to scale the data as shown in:\n https://scikit-learn.org/stable/modules/preprocessing.html
9,10 restarts / seed 137,ConvergenceWarning,lbfgs failed to converge after 24 iteration(s) (status=2):\nABNORMAL: \n\nYou might also want to scale the data as shown in:\n https://scikit-learn.org/stable/modules/preprocessing.html



Function 5 stability checks completed.
Prediction differences are reported in original output units.
Probe points have no observed outputs; these differences are not predictive accuracy measurements.
Warnings may originate from individual optimiser starts, not necessarily the retained best solution.
The current model in gp_models[5] has not been replaced.


### Function 5 stability check: interpretation

The additional fits produced nearly identical log-marginal likelihoods
and fitted kernels. The largest likelihood increase was approximately
0.00000019.

Across the shared probe set, the maximum absolute difference in predicted
means was 0.018816 output units. Local mean differences were smaller,
with RMSE below 0.0003.

These results support retaining the current fit: additional restarts
did not materially change its predicted means on the checked points.
However, the convergence warnings remain documented. This comparison
does not establish predictive accuracy, uncertainty calibration or
stability throughout the entire search space.

The original model remains in `gp_models[5]`.

## 6. Refit auxiliary models

The auxiliary models from CP_23 are refitted using observations through
round 12:

- Function 5: a neural network with one hidden layer of eight units.
- Function 7: linear regression.

Input and target scaling are fitted within each model pipeline.
Predictions are returned in the original output units.

In round 12, the GP had lower absolute error than the neural network
for Function 5. For Function 7, linear regression had lower error than
the GP, although both overpredicted the observed output substantially.
Earlier evaluations showed different relative performance, so this
single result does not establish a consistently superior model.

Auxiliary predictions provide a diagnostic comparison. They do not
supply the uncertainty estimates used by the GP acquisition function.

In [7]:
# CP_24 — Round 13
# Refit auxiliary models using observations through round 12.

import warnings
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

required_variables = ["updated_data", "completed_rounds"]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the data-loading cell first. Missing variables: "
        + ", ".join(missing_variables)
    )

auxiliary_configurations = {
    5: {
        "name": "NN (8,)",
        "estimator": MLPRegressor(
            hidden_layer_sizes=(8,),
            activation="tanh",
            solver="lbfgs",
            alpha=0.1,
            max_iter=2000,
            random_state=42,
        ),
    },
    7: {
        "name": "Linear regression",
        "estimator": LinearRegression(),
    },
}

auxiliary_models = {}
auxiliary_fit_records = []
auxiliary_warning_records = []

for function_id, configuration in auxiliary_configurations.items():
    X = np.asarray(
        updated_data[function_id]["X"],
        dtype=float,
    )
    y = np.asarray(
        updated_data[function_id]["y"],
        dtype=float,
    ).reshape(-1)

    if X.ndim != 2 or X.shape[0] != len(y):
        raise ValueError(
            f"Function {function_id}: incompatible input/output shapes."
        )

    if not np.all(np.isfinite(X)) or not np.all(np.isfinite(y)):
        raise ValueError(
            f"Function {function_id}: non-finite training values."
        )

    input_pipeline = Pipeline([
        ("input_scaler", StandardScaler()),
        ("regressor", configuration["estimator"]),
    ])

    model = TransformedTargetRegressor(
        regressor=input_pipeline,
        transformer=StandardScaler(),
    )

    start_time = perf_counter()

    with warnings.catch_warnings(record=True) as captured_warnings:
        warnings.simplefilter("always")
        model.fit(X, y)

    fit_seconds = perf_counter() - start_time

    auxiliary_models[function_id] = model

    auxiliary_fit_records.append({
        "Function": function_id,
        "Observations": len(y),
        "Auxiliary model": configuration["name"],
        "Fit seconds": fit_seconds,
    })

    for warning in captured_warnings:
        auxiliary_warning_records.append({
            "Function": function_id,
            "Category": warning.category.__name__,
            "Message": str(warning.message),
        })

auxiliary_fit_summary_df = (
    pd.DataFrame(auxiliary_fit_records)
    .set_index("Function")
    .sort_index()
)

auxiliary_warnings_df = pd.DataFrame(
    auxiliary_warning_records,
    columns=["Function", "Category", "Message"],
)

display(
    auxiliary_fit_summary_df.style.format({
        "Fit seconds": "{:.3f}",
    })
)

print(
    "\nAuxiliary models fitted for Functions 5 and 7 "
    f"using observations through round {completed_rounds}."
)
print("Predictions will be returned in the original output units.")

if auxiliary_warnings_df.empty:
    print("No auxiliary fitting warnings were recorded.")
else:
    print(
        f"Recorded auxiliary fitting warnings: "
        f"{len(auxiliary_warnings_df)}"
    )
    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None,
    ):
        display(auxiliary_warnings_df)

,Observations,Auxiliary model,Fit seconds
Function,,,
5,32,"NN (8,)",0.053
7,42,Linear regression,0.015



Auxiliary models fitted for Functions 5 and 7 using observations through round 12.
Predictions will be returned in the original output units.
No auxiliary fitting warnings were recorded.


## 7. Compare candidate policies for the final round

The final evaluation cannot provide information for another submission
within this challenge. Candidate selection therefore needs to consider
immediate performance rather than valuing information only for future use.

I compare three scoring rules on identical candidate pools:

- Predicted mean: prioritises the model's expected output.
- Mean + 1 std: uses a smaller uncertainty bonus.
- Mean + 2 std: retains the previous UCB rule as a reference.

The weights are transparent heuristic comparisons, not tuned or validated
optimal settings. Uncertainty can still matter in the final round because
an uncertain point may exceed the incumbent. Maximising predicted mean
and maximising expected improvement are different objectives; this cell
does not implement Expected Improvement.

For each function, I generate 100,000 global candidates and 10,000
candidates in each of two local boxes around the best observation,
with coordinate half-widths of 0.10 and 0.03.

Candidates are rounded to six decimals before scoring. Duplicate
candidates and points within 0.000001 Euclidean units of existing
observations are excluded. Reproducible seeds vary by function and round.

For Function 5, I also compare proposals requiring at least 0.01
Euclidean distance from every observation. This constraint is evaluated
as an alternative, rather than automatically imposed.

Reported standard deviations include the fitted white-noise component.
Auxiliary predictions provide diagnostic context and do not determine
the candidate scores.

This cell reports the best proposal under each scoring rule across
the three search pools. Final queries have not yet been selected.

In [8]:
# CP_24 — Round 13
# Compare final-round scoring policies on shared candidate pools.

from time import perf_counter

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from IPython.display import display

required_variables = [
    "updated_data",
    "gp_models",
    "auxiliary_models",
    "submission_round",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the preceding notebook cells first. Missing variables: "
        + ", ".join(missing_variables)
    )

candidate_search_settings = {
    "global_candidates": 100_000,
    "local_candidates": 10_000,
    "local_half_widths": (0.10, 0.03),
    "duplicate_tolerance": 1e-6,
    "f5_minimum_distance": 0.01,
    "prediction_batch_size": 5000,
    "base_seed": 42,
}

candidate_policies = {
    "Predicted mean": 0.0,
    "Mean + 1 std": 1.0,
    "Mean + 2 std": 2.0,
}


def format_query(point):
    return "-".join(f"{value:.6f}" for value in point)


def predict_in_batches(model, points, batch_size):
    means = []
    standard_deviations = []

    for start in range(0, len(points), batch_size):
        mean, std = model.predict(
            points[start:start + batch_size],
            return_std=True,
        )
        means.append(mean)
        standard_deviations.append(std)

    mean = np.concatenate(means)
    std = np.concatenate(standard_deviations)

    if (
        not np.all(np.isfinite(mean))
        or not np.all(np.isfinite(std))
        or np.any(std < 0)
    ):
        raise ValueError("Invalid GP predictions in candidate search.")

    return mean, std


proposal_records = []
search_time_records = []
candidate_seeds = {}

for function_id in range(1, 9):
    start_time = perf_counter()

    X = np.asarray(updated_data[function_id]["X"], dtype=float)
    y = np.asarray(
        updated_data[function_id]["y"], dtype=float
    ).reshape(-1)

    dimensions = X.shape[1]
    best_input = X[np.argmax(y)]
    current_best = float(np.max(y))
    observation_tree = cKDTree(X)

    seed = (
        candidate_search_settings["base_seed"]
        + 1000 * submission_round
        + function_id
    )
    candidate_seeds[function_id] = seed
    rng = np.random.default_rng(seed)

    pools = {
        "Global": rng.uniform(
            0.0,
            1.0,
            size=(
                candidate_search_settings["global_candidates"],
                dimensions,
            ),
        )
    }

    for half_width in candidate_search_settings["local_half_widths"]:
        lower = np.maximum(0.0, best_input - half_width)
        upper = np.minimum(1.0, best_input + half_width)

        pools[f"Local {half_width:.2f}"] = rng.uniform(
            lower,
            upper,
            size=(
                candidate_search_settings["local_candidates"],
                dimensions,
            ),
        )

    for search_name, raw_candidates in pools.items():
        # Score the same coordinates that would be submitted.
        candidates = np.unique(
            np.round(raw_candidates, decimals=6),
            axis=0,
        )

        permitted = np.all(
            (candidates >= 0.0) & (candidates < 1.0),
            axis=1,
        )
        candidates = candidates[permitted]

        nearest_distances = observation_tree.query(
            candidates, k=1
        )[0]

        separated = (
            nearest_distances
            > candidate_search_settings["duplicate_tolerance"]
        )
        candidates = candidates[separated]
        nearest_distances = nearest_distances[separated]

        if len(candidates) == 0:
            raise RuntimeError(
                f"No candidates remain for Function {function_id}, "
                f"{search_name}."
            )

        mean, std = predict_in_batches(
            gp_models[function_id],
            candidates,
            candidate_search_settings["prediction_batch_size"],
        )

        proposal_masks = {
            "Standard": np.ones(len(candidates), dtype=bool)
        }

        if function_id == 5:
            proposal_masks["Distance >= 0.01"] = (
                nearest_distances
                >= candidate_search_settings["f5_minimum_distance"]
            )

        for proposal_name, eligible_mask in proposal_masks.items():
            eligible_indices = np.flatnonzero(eligible_mask)

            if len(eligible_indices) == 0:
                raise RuntimeError(
                    f"No eligible candidates for Function {function_id}, "
                    f"{search_name}, {proposal_name}."
                )

            for policy_name, uncertainty_weight in candidate_policies.items():
                scores = mean + uncertainty_weight * std

                winner_index = eligible_indices[
                    np.argmax(scores[eligible_indices])
                ]
                point = candidates[winner_index]

                auxiliary_prediction = np.nan
                if function_id in auxiliary_models:
                    auxiliary_prediction = float(
                        auxiliary_models[function_id].predict(
                            point.reshape(1, -1)
                        )[0]
                    )
                    if not np.isfinite(auxiliary_prediction):
                        raise ValueError(
                            f"Invalid auxiliary prediction for "
                            f"Function {function_id}."
                        )

                proposal_records.append({
                    "Function": function_id,
                    "Policy": policy_name,
                    "Uncertainty weight": uncertainty_weight,
                    "Proposal": proposal_name,
                    "Search": search_name,
                    "Query": format_query(point),
                    "Current best": current_best,
                    "GP mean": float(mean[winner_index]),
                    "GP std": float(std[winner_index]),
                    "Policy score": float(scores[winner_index]),
                    "UCB (mean + 2 std)": float(
                        mean[winner_index] + 2.0 * std[winner_index]
                    ),
                    "Nearest observation distance": float(
                        nearest_distances[winner_index]
                    ),
                    "Distance to best (RMS)": float(
                        np.linalg.norm(point - best_input)
                        / np.sqrt(dimensions)
                    ),
                    "Auxiliary prediction": auxiliary_prediction,
                    "Eligible candidates": len(eligible_indices),
                    "Seed": seed,
                })

    search_time_records.append({
        "Function": function_id,
        "Candidate search seconds": perf_counter() - start_time,
    })

# Keep every search-pool proposal for further inspection.
candidate_comparison_df = pd.DataFrame(proposal_records)

# Best across the three pools for each function, policy and constraint.
winner_indices = (
    candidate_comparison_df
    .groupby(["Function", "Policy", "Proposal"], sort=False)[
        "Policy score"
    ]
    .idxmax()
)

policy_comparison_df = (
    candidate_comparison_df.loc[winner_indices]
    .sort_values(
        ["Function", "Proposal", "Uncertainty weight"],
        kind="stable",
    )
    .reset_index(drop=True)
)

candidate_search_times_df = (
    pd.DataFrame(search_time_records)
    .set_index("Function")
    .sort_index()
)

display_columns = [
    "Function",
    "Policy",
    "Proposal",
    "Search",
    "Query",
    "Current best",
    "GP mean",
    "GP std",
    "Policy score",
    "Nearest observation distance",
    "Auxiliary prediction",
]

numeric_formats = {
    "Current best": "{:.6e}",
    "GP mean": "{:.6e}",
    "GP std": "{:.6e}",
    "Policy score": "{:.6e}",
    "Nearest observation distance": "{:.6f}",
    "Auxiliary prediction": "{:.6e}",
}

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_columns", None,
    "display.max_rows", None,
):
    display(
        policy_comparison_df[display_columns].style.format(
            numeric_formats,
            na_rep="—",
        )
    )

display(
    candidate_search_times_df.style.format({
        "Candidate search seconds": "{:.3f}",
    })
)

print(
    "\nEach policy was evaluated on the same candidate pools "
    "and GP predictions."
)
print(
    "The displayed rows are policy-specific winners across "
    "the global and two local searches."
)
print(
    "Policy scores use different uncertainty weights; "
    "the largest score across policies is not a selection rule."
)
print(
    "Function 5 includes both standard and distance-constrained "
    "alternatives."
)
print(
    "Repeated queries across policies indicate agreement, "
    "not additional evaluations."
)
print("Final round-13 queries have not yet been selected.")

,Function,Policy,Proposal,Search,Query,Current best,GP mean,GP std,Policy score,Nearest observation distance,Auxiliary prediction
0,1,Predicted mean,Standard,Global,0.915579-0.746359,7.710875e-16,1.973928e-04,2.912296e-04,1.973928e-04,0.104554,—
1,1,Mean + 1 std,Standard,Global,0.999616-0.925839,7.710875e-16,-4.609131e-05,6.477085e-04,6.016172e-04,0.076738,—
2,1,Mean + 2 std,Standard,Global,0.006158-0.461500,7.710875e-16,-4.903277e-05,6.496028e-04,1.250173e-03,0.095889,—
3,2,Predicted mean,Standard,Local 0.03,0.697607-0.001246,6.582395e-01,5.678675e-01,6.905136e-02,5.678675e-01,0.001171,—
4,2,Mean + 1 std,Standard,Local 0.03,0.696895-0.000265,6.582395e-01,5.678511e-01,6.908125e-02,6.369324e-01,0.001312,—
5,2,Mean + 2 std,Standard,Local 0.10,0.696247-0.000100,6.582395e-01,5.677758e-01,6.912092e-02,7.060176e-01,0.001059,—
6,3,Predicted mean,Standard,Global,0.468809-0.987323-0.072474,-3.096391e-02,-2.719832e-02,2.443408e-02,-2.719832e-02,0.236807,—
7,3,Mean + 1 std,Standard,Global,0.053450-0.994619-0.076970,-3.096391e-02,-3.576217e-02,3.813586e-02,2.373690e-03,0.327820,—
8,3,Mean + 2 std,Standard,Global,0.988217-0.005939-0.377631,-3.096391e-02,-5.746184e-02,5.081867e-02,4.417550e-02,0.283851,—
9,4,Predicted mean,Standard,Local 0.03,0.405875-0.414442-0.381493-0.418806,6.152729e-01,4.802052e-01,1.469240e-01,4.802052e-01,0.004678,—


,Candidate search seconds
Function,
1,0.679
2,0.379
3,0.468
4,0.621
5,0.479
6,0.489
7,0.617
8,0.671



Each policy was evaluated on the same candidate pools and GP predictions.
The displayed rows are policy-specific winners across the global and two local searches.
Policy scores use different uncertainty weights; the largest score across policies is not a selection rule.
Function 5 includes both standard and distance-constrained alternatives.
Repeated queries across policies indicate agreement, not additional evaluations.
Final round-13 queries have not yet been selected.


## 8. Final-round selection and validation

For the final submission, I select the standard candidate with the
highest GP predictive mean across the global and two local search pools.

This is an explicit change from the previous mean-plus-two-standard-
deviations rule. It prioritises predicted immediate output because
information obtained from the final evaluation cannot guide another
submission within this challenge.

This policy is a heuristic, not a validated optimum. Maximising expected
output is different from maximising expected improvement over the
incumbent. Uncertain candidates could still produce better realised
results, and the GP predictions may be inaccurate.

### Evidence informing the decision

For Functions 4, 5 and 6, predicted mean and mean plus one standard
deviation select the same candidate. For Function 7, the mean-based
candidate lies in the narrower local search and has a higher predicted
mean than either uncertainty-bonus winner. Its actual outcome remains
unknown, particularly given the substantial round-12 prediction error.

For Function 5, I remove the additional minimum distance of 0.01.
The best standard mean-based proposal predicts approximately 8597.128,
whereas the best distance-constrained alternative predicts 8297.894.
The general duplicate tolerance remains in place. Neither the small GP
standard deviation nor agreement between scoring rules guarantees success;
the neural network predicts a lower output.

Functions 1 and 3 retain proposals from the global pool. Exploitation of
a surrogate prediction does not necessarily imply geographical proximity
to the best observation.

For Functions 2 and 6, even the highest candidate mean is below the
incumbent. Function 2 also proposes a point close to existing observations.
These limitations are recorded rather than interpreted as predicted gains.

### Validation and reporting

Each query is checked for dimensionality, finite coordinates, permitted
bounds, six-decimal formatting and separation from previous observations.
Predictions and uncertainty are retained for comparison with final feedback.

Only the coordinate strings are submitted to the portal.
No final-round improvement is claimed before the evaluations are returned.

In [9]:
# CP_24 — Round 13
# Select the highest predicted mean among standard proposals.
# Validate and print the eight final queries.

import re

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from IPython.display import display

required_variables = [
    "candidate_comparison_df",
    "candidate_search_settings",
    "updated_data",
    "gp_models",
    "submission_round",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the preceding notebook cells first. Missing variables: "
        + ", ".join(missing_variables)
    )

if submission_round != 13:
    raise ValueError("This cell prepares the final submission, round 13.")

final_selection_policy = "Predicted mean"
final_proposal_type = "Standard"

duplicate_tolerance = float(
    candidate_search_settings["duplicate_tolerance"]
)

selected_records = []
validation_records = []
selected_queries = {}

for function_id in range(1, 9):
    eligible = candidate_comparison_df.loc[
        (candidate_comparison_df["Function"] == function_id)
        & (candidate_comparison_df["Policy"] == final_selection_policy)
        & (candidate_comparison_df["Proposal"] == final_proposal_type)
    ].copy()

    if eligible.empty:
        raise RuntimeError(
            f"No eligible proposal for Function {function_id}."
        )

    if not np.all(
        np.isfinite(eligible["GP mean"].to_numpy(dtype=float))
    ):
        raise ValueError(
            f"Non-finite candidate means for Function {function_id}."
        )

    # Full stored precision determines selection.
    winner = eligible.loc[eligible["GP mean"].idxmax()].copy()

    X = np.asarray(updated_data[function_id]["X"], dtype=float)
    y = np.asarray(
        updated_data[function_id]["y"], dtype=float
    ).reshape(-1)

    dimensions = X.shape[1]
    query_string = str(winner["Query"])

    pattern = (
        r"0\.\d{6}(?:-0\.\d{6}){"
        + str(dimensions - 1)
        + r"}"
    )

    if re.fullmatch(pattern, query_string) is None:
        raise ValueError(
            f"Invalid query format for Function {function_id}: "
            f"{query_string}"
        )

    query = np.array(
        [float(value) for value in query_string.split("-")],
        dtype=float,
    )

    if query.shape != (dimensions,):
        raise ValueError(
            f"Incorrect dimensionality for Function {function_id}."
        )

    if (
        not np.all(np.isfinite(query))
        or np.any(query < 0.0)
        or np.any(query >= 1.0)
    ):
        raise ValueError(
            f"Invalid coordinates for Function {function_id}."
        )

    nearest_distance = float(cKDTree(X).query(query, k=1)[0])

    if nearest_distance <= duplicate_tolerance:
        raise ValueError(
            f"Function {function_id}: query violates duplicate tolerance."
        )

    # Confirm that the retained model reproduces the recorded predictions.
    checked_mean, checked_std = gp_models[function_id].predict(
        query.reshape(1, -1),
        return_std=True,
    )

    checked_mean = float(checked_mean[0])
    checked_std = float(checked_std[0])

    if not np.all(np.isfinite([checked_mean, checked_std])):
        raise ValueError(
            f"Non-finite predictions for Function {function_id}."
        )

    if checked_std < 0:
        raise ValueError(
            f"Negative predictive standard deviation for Function "
            f"{function_id}."
        )

    if not np.isclose(
        checked_mean, float(winner["GP mean"]),
        rtol=1e-7, atol=1e-8,
    ) or not np.isclose(
        checked_std, float(winner["GP std"]),
        rtol=1e-7, atol=1e-8,
    ):
        raise RuntimeError(
            f"Function {function_id}: predictions changed after candidate "
            "generation. Rerun the candidate-comparison cell."
        )

    current_best = float(np.max(y))

    selected_records.append({
        "Function": function_id,
        "Selected policy": final_selection_policy,
        "Selected search": winner["Search"],
        "Proposal": final_proposal_type,
        "Query": query_string,
        "Current best": current_best,
        "Predicted mean": checked_mean,
        "Predicted std": checked_std,
        "Predicted mean minus current best": (
            checked_mean - current_best
        ),
        "UCB (mean + 2 std)": checked_mean + 2.0 * checked_std,
        "Nearest observation distance": nearest_distance,
        "Auxiliary prediction": float(winner["Auxiliary prediction"]),
    })

    selected_queries[function_id] = query.copy()

    validation_records.append({
        "Function": function_id,
        "Dimensions": dimensions,
        "Format valid": True,
        "Bounds valid": True,
        "Duplicate tolerance satisfied": True,
        "Nearest observation distance": nearest_distance,
    })

selected_candidates_df = (
    pd.DataFrame(selected_records)
    .set_index("Function")
    .sort_index()
)

submission_queries_df = (
    selected_candidates_df[["Query"]]
    .rename(columns={"Query": "Query for submission"})
    .copy()
)

query_validation_df = (
    pd.DataFrame(validation_records)
    .set_index("Function")
    .sort_index()
)

# Retain the prediction snapshot for later comparison with final feedback.
round13_predictions_df = selected_candidates_df.copy()

formats = {
    "Current best": "{:.6e}",
    "Predicted mean": "{:.6e}",
    "Predicted std": "{:.6e}",
    "Predicted mean minus current best": "{:.6e}",
    "UCB (mean + 2 std)": "{:.6e}",
    "Nearest observation distance": "{:.6f}",
    "Auxiliary prediction": "{:.6e}",
}

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_columns", None,
):
    display(
        selected_candidates_df.style.format(
            formats,
            na_rep="—",
        )
    )
    display(submission_queries_df)

print(f"\nRound {submission_round} — final queries for manual submission\n")

for function_id, row in submission_queries_df.iterrows():
    print(
        f"Function {function_id}: {row['Query for submission']}"
    )

print("\nSelection policy: highest predicted mean, standard proposals.")
print(
    "The additional F5 distance constraint of 0.01 is not imposed."
)
print("The general duplicate tolerance remains in force.")
print("\nAll eight queries passed validation.")
print(
    "Predicted mean minus current best is a model-based difference, "
    "not observed improvement or Expected Improvement."
)
print(
    "Queries are prepared; this code does not submit them to the portal."
)

,Selected policy,Selected search,Proposal,Query,Current best,Predicted mean,Predicted std,Predicted mean minus current best,UCB (mean + 2 std),Nearest observation distance,Auxiliary prediction
Function,,,,,,,,,,,
1,Predicted mean,Global,Standard,0.915579-0.746359,7.710875e-16,1.973928e-04,2.912296e-04,1.973928e-04,7.798521e-04,0.104554,—
2,Predicted mean,Local 0.03,Standard,0.697607-0.001246,6.582395e-01,5.678675e-01,6.905136e-02,-9.037197e-02,7.059703e-01,0.001171,—
3,Predicted mean,Global,Standard,0.468809-0.987323-0.072474,-3.096391e-02,-2.719832e-02,2.443408e-02,3.765598e-03,2.166984e-02,0.236807,—
4,Predicted mean,Local 0.03,Standard,0.405875-0.414442-0.381493-0.418806,6.152729e-01,4.802052e-01,1.469240e-01,-1.350678e-01,7.740532e-01,0.004678,—
5,Predicted mean,Local 0.03,Standard,0.999129-0.999884-0.999390-0.998665,8.515379e+03,8.597128e+03,4.974057e+00,8.174905e+01,8.607076e+03,0.002834,8.263272e+03
6,Predicted mean,Local 0.10,Standard,0.474875-0.376314-0.563286-0.723873-0.001016,-2.424310e-01,-3.627182e-01,9.149107e-02,-1.202872e-01,-1.797360e-01,0.025270,—
7,Predicted mean,Local 0.03,Standard,0.042573-0.233262-0.499894-0.161327-0.302051-0.698182,2.436323e+00,2.470925e+00,3.775303e-02,3.460177e-02,2.546431e+00,0.037815,1.209424e+00
8,Predicted mean,Local 0.10,Standard,0.107547-0.163849-0.135675-0.133563-0.973583-0.505314-0.241550-0.834686,9.958227e+00,9.964779e+00,9.286254e-03,6.552175e-03,9.983351e+00,0.108134,—


,Query for submission
Function,
1,0.915579-0.746359
2,0.697607-0.001246
3,0.468809-0.987323-0.072474
4,0.405875-0.414442-0.381493-0.418806
5,0.999129-0.999884-0.999390-0.998665
6,0.474875-0.376314-0.563286-0.723873-0.001016
7,0.042573-0.233262-0.499894-0.161327-0.302051-0.698182
8,0.107547-0.163849-0.135675-0.133563-0.973583-0.505314-0.241550-0.834686



Round 13 — final queries for manual submission

Function 1: 0.915579-0.746359
Function 2: 0.697607-0.001246
Function 3: 0.468809-0.987323-0.072474
Function 4: 0.405875-0.414442-0.381493-0.418806
Function 5: 0.999129-0.999884-0.999390-0.998665
Function 6: 0.474875-0.376314-0.563286-0.723873-0.001016
Function 7: 0.042573-0.233262-0.499894-0.161327-0.302051-0.698182
Function 8: 0.107547-0.163849-0.135675-0.133563-0.973583-0.505314-0.241550-0.834686

Selection policy: highest predicted mean, standard proposals.
The additional F5 distance constraint of 0.01 is not imposed.
The general duplicate tolerance remains in force.

All eight queries passed validation.
Predicted mean minus current best is a model-based difference, not observed improvement or Expected Improvement.
Queries are prepared; this code does not submit them to the portal.


## 9. Final-round decisions and limitations

For round 13, I selected the highest GP predicted mean among the eligible candidates from the global and two local searches. I compared uncertainty weights of zero, one and two using the same candidate pools, then chose a mean-based policy for the final submission.

This decision prioritises predicted immediate performance because there is no subsequent query round in which to use newly acquired information. It is a practical heuristic, not proof of an optimal final-round policy. Maximising predicted mean is also different from maximising Expected Improvement above the best observed output.

### Interpretation of the selected queries

- **Function 1:** The selected candidate comes from the global search. Its positive predicted mean remains uncertain, especially given the extremely small observed outputs and the absence of improvement over the initial best.
- **Function 2:** The selected local candidate is close to an existing observation. Its predicted mean remains below the current best, so the model does not predict an improvement in mean.
- **Function 3:** The global candidate has a slightly higher predicted mean than the current best. This remains an uncertain prediction rather than evidence of improvement.
- **Function 4:** The selected point refines the previously promising local region. The predicted-mean and mean-plus-one-standard-deviation policies selected the same point, although its predicted mean remains below the current best.
- **Function 5:** I removed the additional minimum-distance constraint of 0.01 while retaining the general duplicate tolerance. All three policies selected the same standard candidate, whose predicted mean is approximately 81.75 above the current best. However, the neural-network prediction is lower, and numerical stability of the GP fit does not establish predictive accuracy or uncertainty calibration.
- **Function 6:** All three policies selected the same candidate. Its predicted mean remains below the current best, illustrating that agreement between policies does not guarantee a promising outcome.
- **Function 7:** After the large prediction error in round 12, I selected a local candidate using predicted mean without an uncertainty bonus. The GP and linear model still disagree substantially, so this choice remains model-dependent.
- **Function 8:** The selected local candidate has a predicted mean approximately 0.00655 above the current best. The potential gain is modest and must be checked against the actual evaluation.

### Feedback and final evaluation

Neither round 11 nor round 12 improved the best observed outputs. These results motivated reviewing the selection policy, but they do not establish convergence or prove that exploration was ineffective.

Once round-13 feedback is available, I will compare the observed outputs with the recorded predictions, update the best-so-far results and document whether the final decisions produced improvements.

All eight queries passed the implemented validation checks. They are prepared for manual portal submission; no round-13 outcomes are available in this notebook yet.